# Efficient Collections and Data Structures in Python

This notebook is a short guide to Python's **collections** and how to use them
to build efficient data structures (queues, stacks, graphs, caches, etc.).

You already know basic Python, so the focus here is on **which container to use
for which pattern**, and the **time complexity** of common operations.


### Contents

- [1. Core Ideas: Containers and Complexity](#1-core-ideas-containers-and-complexity)
- [2. `deque`: Fast Queues and Stacks](#2-deque-fast-queues-and-stacks)
- [3. `defaultdict` and `Counter`](#3-defaultdict-and-counter)
- [4. Ordered Mappings and LRU Cache](#4-ordered-mappings-and-lru-cache)
- [5. Heaps and Priority Queues](#5-heaps-and-priority-queues)
- [6. Sorted Sequences: `bisect` and Third‑Party Helpers](#6-sorted-sequences-bisect-and-third-party-helpers)
- [7. Graphs and Trees with Built‑In Collections](#7-graphs-and-trees-with-built-in-collections)
- [8. `networkx` (Optional, Third‑Party)](#7-networkx-optional-third-party)
- [9. Sorted Containers (Third‑Party)](#9-sorted-containers-third-party)


## 1. Core Ideas: Containers and Complexity

Rough complexity (average case):

- `list.append`, `list.pop()` at the end: **O(1)** amortized.
- `list.pop(0)`: **O(n)** (shifts all elements).
- `x in list`: **O(n)**.
- `x in set`, `x in dict`: **O(1)** average (hash lookup).
- `deque.append`, `deque.appendleft`, `deque.pop`, `deque.popleft`: **O(1)**.

These determine whether a structure will scale or not.


In [ ]:
# 1.1 Tiny timing: list.pop(0) vs deque.popleft()

from collections import deque
import time

def time_pop_front_list(n: int) -> float:
    data = list(range(n))
    t0 = time.perf_counter()
    while data:
        data.pop(0)
    return time.perf_counter() - t0

def time_popleft_deque(n: int) -> float:
    data = deque(range(n))
    t0 = time.perf_counter()
    while data:
        data.popleft()
    return time.perf_counter() - t0

for n in (10_000, 50_000):
    t_list = time_pop_front_list(n)
    t_deque = time_popleft_deque(n)
    print(f"n={n:6d} list.pop(0): {t_list:.4f}s, deque.popleft(): {t_deque:.4f}s")

n= 10000 list.pop(0): 0.0753s, deque.popleft(): 0.0002s
n= 50000 list.pop(0): 1.8929s, deque.popleft(): 0.0008s


## 2. `deque`: Fast Queues and Stacks

`collections.deque` is a double-ended queue with O(1) appends and pops from
both ends. Typical patterns:

- FIFO queue: `append` then `popleft`.
- Stack: `append` then `pop`.
- Sliding window: `deque(maxlen=N)`.


In [ ]:
# 2.1 Queue and stack with deque

from collections import deque

q = deque()
for i in range(3):
    q.append(i)
print("queue start:", list(q))
while q:
    print("popleft ->", q.popleft())

s = deque()
for ch in ["A", "B", "C"]:
    s.append(ch)
print("stack start:", list(s))
while s:
    print("pop ->", s.pop())

queue start: [0, 1, 2]
popleft -> 0
popleft -> 1
popleft -> 2
stack start: ['A', 'B', 'C']
pop -> C
pop -> B
pop -> A


## 3. `defaultdict` and `Counter`

- `defaultdict(list)` is ideal for **grouping** and **adjacency lists**.
- `Counter` is ideal for **frequency counting**.


In [ ]:
# 3.1 Grouping with defaultdict(list)

from collections import defaultdict, Counter

pairs = [("even", 2), ("odd", 1), ("even", 4), ("odd", 3)]
groups: defaultdict[str, list[int]] = defaultdict(list)
for k, v in pairs: groups[k].append(v)
print(dict(groups))

# 3.2 Counting with Counter
data = [1, 2, 1, 3, 2, 1, 4]
freq = Counter(data)
print(freq)
print("most common:", freq.most_common(2))

{'even': [2, 4], 'odd': [1, 3]}
Counter({1: 3, 2: 2, 3: 1, 4: 1})
most common: [(1, 3), (2, 2)]


## 4. Ordered Mappings and LRU Cache

Modern `dict` preserves insertion order, but `collections.OrderedDict` gives
extra methods like `move_to_end` and `popitem(last=False)` which are handy for
building LRU caches.


In [4]:
# 4.1 Very small LRU cache using OrderedDict

from collections import OrderedDict
from typing import Hashable, Any

class LRUCache:
    def __init__(self, capacity: int):
        self.capacity = capacity
        self.data: OrderedDict[Hashable, Any] = OrderedDict()

    def get(self, key: Hashable, default: Any = None) -> Any:
        if key not in self.data:
            return default
        self.data.move_to_end(key)
        return self.data[key]

    def put(self, key: Hashable, value: Any) -> None:
        if key in self.data:
            self.data.move_to_end(key)
        self.data[key] = value
        if len(self.data) > self.capacity:
            self.data.popitem(last=False)

cache = LRUCache(2)
cache.put("a", 1)
cache.put("b", 2)
print("get a:", cache.get("a"))
cache.put("c", 3)  # evicts b
print("keys:", list(cache.data.keys()))


get a: 1
keys: ['a', 'c']


## 5. Heaps and Priority Queues

`heapq` implements a **min-heap** on top of a plain Python list. The **heap
object** is just that list, and the `heapq` functions are its "methods":

- `heapq.heapify(heap)`: turn a list into a valid min-heap in-place (O(n)).
- `heapq.heappush(heap, x)`: push a new item (O(log n)).
- `heapq.heappop(heap)`: pop and return the smallest item (O(log n)).
- `heapq.heappushpop(heap, x)`: push, then pop the smallest (useful for fixed-size heaps).
- `heapq.heapreplace(heap, x)`: pop smallest, then push `x`.
- `heapq.nsmallest(k, iterable)` / `heapq.nlargest(k, iterable)`: helpers for top-k queries.

We usually wrap these primitives to model a **priority queue** or other
heap-based data structure.


In [14]:
# 5.1 Heap as an object + priority queue

import heapq
from typing import Generic, TypeVar, List, Tuple

T = TypeVar("T")

class MinHeap(Generic[T]):
    """Small wrapper around a list + heapq.

    The underlying list is the heap object; heapq.* are its "methods".
    """

    def __init__(self, items: List[T] | None = None) -> None:
        self._data: List[T] = [] if items is None else list(items)
        heapq.heapify(self._data)

    def push(self, item: T) -> None:
        heapq.heappush(self._data, item)

    def pop(self) -> T:
        return heapq.heappop(self._data)

    def peek(self) -> T:
        return self._data[0]

    def __len__(self) -> int:  # len(heap)
        return len(self._data)

    def to_list(self) -> List[T]:
        return list(self._data)


# Priority queue of (priority, name) using MinHeap
Pair = Tuple[int, str]
pq = MinHeap[Pair]()
pq.push((3, "low"))
pq.push((1, "high"))
pq.push((2, "medium"))

while pq:
    priority, name = pq.pop()
    print(priority, name)

1 high
2 medium
3 low


## 6. Graphs and Trees with Built‑In Collections

You can model **graphs** and **trees** with just dicts, lists, and `deque`:

- Undirected graph: `dict[node, list[neighbor]]` or `defaultdict(list)`.
- Directed graph: same, but edges are one‑way.
- Tree: nested nodes (e.g. small `Node` records) or parent → children adjacency.

**BFS (Breadth‑First Search)**: explores the graph in "layers" from a start node. It visits all neighbors at distance 1, then distance 2, and so on. BFS typically uses a queue (`deque`) and is ideal for finding shortest paths in unweighted graphs (fewest edges).

**DFS (Depth‑First Search)**: explores by going as deep as possible along one path before backtracking. DFS typically uses a stack (explicit list) or recursion and is useful for tasks like reachability, cycle detection, and building traversal orders.

**Why are they useful**:

- **You use BFS/DFS when your data is a network** (roads, dependencies, links, friends, file imports, states in a puzzle). A graph/tree is just “things + connections”; BFS/DFS are the standard ways to *systematically explore* those connections.
- **BFS answers "closest" questions**: *What is the minimum number of steps from A to B?* (unweighted shortest path), *Which nodes are within 2 hops?*.
- **DFS answers "structure" questions**: *Is everything reachable?*, *Is there a cycle?*, *In what order should I process nodes?* (topological-style reasoning for DAGs).
- Without BFS/DFS you end up writing ad-hoc exploration loops that are easy to get wrong (infinite loops, missed nodes). These algorithms are the reliable building blocks.

Below is a simple unweighted graph using an adjacency list plus BFS and DFS.


In [16]:
# 7.1 Graph with adjacency list + BFS / DFS

from collections import defaultdict, deque
from typing import Dict, List

Graph = Dict[str, list[str]]

def add_edge(graph: Graph, u: str, v: str) -> None:
    graph[u].append(v)
    graph[v].append(u)

def bfs(graph: Graph, start: str) -> list[str]:
    visited = {start}
    order: list[str] = []
    q: deque[str] = deque([start])
    while q:
        u = q.popleft()
        order.append(u)
        for v in graph[u]:
            if v not in visited:
                visited.add(v)
                q.append(v)
    return order

def dfs(graph: Graph, start: str) -> list[str]:
    visited = set()
    order: list[str] = []
    stack: list[str] = [start]
    while stack:
        u = stack.pop()
        if u in visited:
            continue
        visited.add(u)
        order.append(u)
        for v in reversed(graph[u]):
            if v not in visited:
                stack.append(v)
    return order

g: Graph = defaultdict(list)
add_edge(g, "A", "B")
add_edge(g, "A", "C")
add_edge(g, "B", "D")
add_edge(g, "C", "E")

print("BFS from A:", bfs(g, "A"))
print("DFS from A:", dfs(g, "A"))

BFS from A: ['A', 'B', 'C', 'D', 'E']
DFS from A: ['A', 'B', 'D', 'C', 'E']


## 7. `networkx` (Optional, Third‑Party)

[`networkx`](https://networkx.org/) is a **very popular** library for working
with graphs in Python. It lets you build graphs using familiar containers
under the hood, but gives you algorithms (shortest paths, centrality, etc.)
out of the box.

It is **not** in the standard library, but widely used. You can install it
with `pip install networkx`.


In [8]:
# 8.1 Tiny networkx demo (if installed)

try:
    import networkx as nx
except ImportError as e:
    print("networkx not installed; run 'pip install networkx' to try this demo.")
else:
    # Simple unweighted shortest path
    G = nx.Graph()
    G.add_edge("A", "B")
    G.add_edge("A", "C")
    G.add_edge("B", "D")
    print("nodes:", list(G.nodes()))
    print("edges:", list(G.edges()))
    print("shortest path A->D:", nx.shortest_path(G, "A", "D"))

    # Weighted shortest path on a directed graph
    DG = nx.DiGraph()
    edges = [
        ("A", "B", 2.0),
        ("A", "C", 1.0),
        ("C", "B", 0.5),
        ("B", "D", 1.0),
        ("C", "D", 4.0),
    ]
    for u, v, w in edges:
        DG.add_edge(u, v, weight=w)
    w_path = nx.dijkstra_path(DG, "A", "D", weight="weight")
    w_dist = nx.dijkstra_path_length(DG, "A", "D", weight="weight")
    print("weighted shortest path A->D:", w_path, "distance=", w_dist)

    # Basic centrality measures
    deg = nx.degree_centrality(G)
    bet = nx.betweenness_centrality(G)
    print("degree centrality:", deg)
    print("betweenness centrality:", bet)


nodes: ['A', 'B', 'C', 'D']
edges: [('A', 'B'), ('A', 'C'), ('B', 'D')]
shortest path A->D: ['A', 'B', 'D']


## 8. Sorted Sequences: `bisect` and Third‑Party Helpers

Sometimes you want to keep a **sorted list** and do fast lookups/insertions
by value. Python's built‑in `bisect` module does binary search on plain lists.

- `bisect_left(a, x)` / `bisect_right(a, x)` give insertion positions in O(log n).
- `insort_left(a, x)` / `insort_right(a, x)` insert while keeping the list sorted
  (overall cost O(n) because of shifting, but search is O(log n)).

In [11]:
# 6.1 Using bisect to maintain a sorted list

import bisect

values: list[int] = []

for x in [5, 1, 4, 2, 3]:
    bisect.insort(values, x)
    print("after inserting", x, "->", values)

# find position where 3 would be inserted
pos = bisect.bisect_left(values, 3)
print("bisect_left(3) ->", pos)


after inserting 5 -> [5]
after inserting 1 -> [1, 5]
after inserting 4 -> [1, 4, 5]
after inserting 2 -> [1, 2, 4, 5]
after inserting 3 -> [1, 2, 3, 4, 5]
bisect_left(3) -> 2


## 9. Sorted Containers (Third‑Party)

The standard library gives you `bisect` (binary search on lists) and
`heapq` (priority queues). When you need **fully-fledged sorted data
structures** that stay sorted automatically and support efficient
operations, a very popular choice is the third‑party package
[`sortedcontainers`](https://github.com/grantjenks/python-sortedcontainers).

It provides three main types:

- `SortedList`: a list that stays sorted on insert/remove; supports
  indexing, slicing, and `bisect`-like operations in O(log n) (plus O(n)
  for shifting, but heavily optimized in C/Python).
- `SortedDict`: a mapping ordered by key; supports fast lookups by key
  and efficient iteration over key ranges (e.g. `irange` for
  `lo <= key <= hi`).
- `SortedSet`: a sorted, duplicate-free set.

These are very handy when you need:

- Order statistics ("k-th smallest"),
- Range queries over keys or values,
- Data structures like order books, interval maps, or time-indexed
  caches, without re-implementing balanced trees yourself.

You would typically `pip install sortedcontainers` and then import
`SortedList` / `SortedDict` in the specific module where you need them.

In [ ]:
# 9.1 Examples with SortedList, SortedDict, and SortedSet (if installed)

try:
    from sortedcontainers import SortedList, SortedDict, SortedSet
except ImportError:
    print("sortedcontainers not installed; run 'pip install sortedcontainers' to try these demos.")
else:
    # SortedList example: keep numbers in order and do range queries
    prices = SortedList([10, 5, 7])
    prices.add(8)
    print("SortedList prices:", list(prices))
    # All prices between 6 and 9 (inclusive of 6, exclusive of 9)
    lo = prices.bisect_left(6)
    hi = prices.bisect_left(9)
    print("6 <= price < 9:", list(prices[lo:hi]))

    # SortedDict example: keys are kept sorted
    order_book = SortedDict()
    order_book[101.5] = "sell 2"
    order_book[100.0] = "sell 5"
    order_book[102.0] = "sell 1"
    print("SortedDict levels:", list(order_book.items()))
    # Best (lowest) ask:
    best_price = order_book.peekitem(0)[0]
    print("best ask:", best_price)

    # SortedSet example: sorted, unique elements
    symbols = SortedSet(["BTC", "ETH", "BTC", "XRP"])
    print("SortedSet symbols:", list(symbols))
    # iterate over a range of keys
    print("symbols between 'DOGE' and 'Z':", list(symbols.irange("DOGE", "Z")))

SortedList prices: [5, 7, 8, 10]
6 <= price < 9: [7, 8]
SortedDict levels: [(100.0, 'sell 5'), (101.5, 'sell 2'), (102.0, 'sell 1')]
best ask: 100.0
SortedSet symbols: ['BTC', 'ETH', 'XRP']
symbols between 'DOGE' and 'Z': ['ETH', 'XRP']
